<a href="https://colab.research.google.com/github/William-franklyn/Music-recommendation-system-with-AI/blob/main/music_recommendation_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install fastapi uvicorn
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

import re, time, threading, subprocess, requests, torch, uvicorn
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM
from fastapi import FastAPI
from pydantic import BaseModel

HF_TOKEN = userdata.get("HF_TOKEN")
KV_URL = userdata.get("KV_REST_API_URL")
KV_TOKEN = userdata.get("KV_REST_API_TOKEN")
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
REDIS_KEY = "vibefinder_llama_url"

# 1. Load Llama onto the T4 GPU
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, token=HF_TOKEN, torch_dtype=torch.float16, device_map="cuda")
print("Llama loaded")

# 2. A small web server so the app can talk to Llama
app = FastAPI()
lock = threading.Lock()  # one answer at a time, so the GPU is not overloaded

class ChatRequest(BaseModel):
    messages: list
    max_tokens: int = 400

@app.get("/health")
def health():
    return {"ok": True}

@app.post("/chat")
def chat(req: ChatRequest):
    ids = tok.apply_chat_template(req.messages, add_generation_prompt=True,
                                  return_tensors="pt").to("cuda")
    with lock, torch.no_grad():
        out = model.generate(ids, max_new_tokens=min(req.max_tokens, 800),
                             do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    return {"response": text}

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000,
                 log_level="warning"), daemon=True).start()

# 3. Cloudflare tunnel gives the server a public link
proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000",
                         "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
public_url = None
for line in proc.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in proc.stderr], daemon=True).start()
print("Public link:", public_url)

# 4. Save the link in Redis. It expires after 2 minutes unless refreshed,
#    so when Colab stops, the app switches to Cloudflare by itself.
def save_link():
    requests.post(KV_URL, headers={"Authorization": f"Bearer {KV_TOKEN}"},
                  json=["SET", REDIS_KEY, public_url, "EX", "120"], timeout=10)

print("Running. Keep this cell going. Stop it to switch the app to Cloudflare.")
while True:
    try:
        save_link()
    except Exception as e:
        print("Redis error:", e)
    time.sleep(60)